# 27 — Residuals, Normalization, and Regularization

**Master syllabus coverage:** V2 2.14, 3.9–3.10

## Why this matters

Normalization and residuals make deep optimization practical; regularization and honest validation determine whether fitted behavior survives beyond the training examples.

## Learning objectives

- Implement LayerNorm over the correct feature axis.
- Explain residual gradient flow through an identity path.
- Distinguish dropout, weight decay, and early stopping.
- Diagnose underfitting, overfitting, and instability from evidence.

Work through the notebook in order. Predict shapes and results before running each code cell. An assertion failure is part of the lesson: read it before changing code.


## 1. Normalization standardizes a representation

LayerNorm operates independently at each token across its feature dimension:

$$\hat x=\frac{x-\mu}{\sqrt{\sigma^2+\epsilon}},\qquad y=\gamma\hat x+\beta.$$

It does not mix batch items or time steps. RMSNorm removes mean subtraction and normalizes
root mean square. Both learn a scale; LayerNorm also learns a shift.


In [ ]:
import copy
import torch
import torch.nn.functional as F

torch.manual_seed(42)
x = torch.randn(2, 3, 8) * 4 + 7  # [B,T,C]

def manual_layer_norm(value, eps=1e-5):
    mean = value.mean(dim=-1, keepdim=True)
    variance = value.var(dim=-1, unbiased=False, keepdim=True)
    return (value - mean) / torch.sqrt(variance + eps)

normalized = manual_layer_norm(x)
reference = F.layer_norm(x, (x.shape[-1],))
torch.testing.assert_close(normalized, reference)
torch.testing.assert_close(normalized.mean(dim=-1), torch.zeros(2, 3), atol=1e-6, rtol=0)
print("per-token mean and variance normalized")


## 2. Residual paths create an identity route

A residual block computes `x + F(x)`. Its Jacobian is `I + J_F`, so gradients have a
direct identity contribution. Residuals do not guarantee perfect optimization, but they
make learning incremental corrections easier. Transformer pre-norm blocks normalize
before each sublayer and add its result to the residual stream.


In [ ]:
def input_gradient(depth: int, residual: bool) -> float:
    torch.manual_seed(0)
    value = torch.randn(16, 32, requires_grad=True)
    original = value
    for _ in range(depth):
        weight = torch.randn(32, 32) * 0.03
        transformed = torch.tanh(value @ weight)
        value = value + transformed if residual else transformed
    value.square().mean().backward()
    return original.grad.norm().item()

for residual in (False, True):
    print("residual=", residual, "input grad norm=", input_gradient(20, residual))


## 3. Generalization tools solve different problems

- **Weight decay** prefers smaller parameter values.
- **Dropout** injects multiplicative noise during training and rescales survivors.
- **Early stopping** selects a checkpoint before validation performance degrades.
- **More/better data** changes the evidence and is often the strongest intervention.

These tools do not repair leakage, mislabeled data, an inappropriate metric, or an
optimization failure.


In [ ]:
dropout = torch.nn.Dropout(p=0.5)
ones = torch.ones(20_000)
dropout.train()
train_output = dropout(ones)
dropout.eval()
eval_output = dropout(ones)
print("train mean:", train_output.mean().item(), "zero fraction:", (train_output == 0).float().mean().item())
print("eval unique:", eval_output.unique().tolist())
assert abs(train_output.mean().item() - 1.0) < 0.05
assert torch.equal(eval_output, ones)


## 4. Diagnose with train/validation curves

Underfitting: both losses remain high. Overfitting: training improves while validation
worsens. Optimization instability: both curves spike or become non-finite. Distribution
shift: validation may stay persistently worse even when regularization is strong.


In [ ]:
def diagnose(train_loss, val_loss):
    if not (torch.isfinite(torch.tensor(train_loss)).all() and torch.isfinite(torch.tensor(val_loss)).all()):
        return "numerical/optimization instability"
    if train_loss[-1] > 0.8 and val_loss[-1] > 0.8:
        return "underfitting or optimization failure"
    if train_loss[-1] < train_loss[0] and val_loss[-1] > min(val_loss):
        return "overfitting after best checkpoint"
    return "no obvious pathology from loss curves alone"

examples = {
    "underfit": ([1.2, 1.1, 1.0], [1.3, 1.2, 1.1]),
    "overfit": ([1.0, 0.5, 0.1], [1.1, 0.7, 0.9]),
    "stable": ([1.0, 0.6, 0.4], [1.1, 0.7, 0.5]),
}
for name, curves in examples.items():
    print(name, "→", diagnose(*curves))


## Failure modes to recognize

- **Normalization over batch/time:** examples or tokens leak into each other's statistics.
- **Residual shape mismatch:** the identity and update paths cannot be added.
- **Dropout active during evaluation:** metrics and generation remain stochastic for the wrong reason.
- **Regularizing an optimization bug:** lower capacity cannot fix broken gradients or learning rates.

These are diagnostic patterns, not trivia. When a future model fails, reduce the problem to the smallest example in this notebook and test the relevant invariant.


## Deliberate practice

1. Implement RMSNorm and compare its output statistics and parameter count with LayerNorm.
2. Measure input-gradient norms across depth with plain, residual, and pre-norm residual blocks.
3. Given three pairs of learning curves, propose one diagnostic experiment before selecting a fix.

Do not merely rerun the provided cells. Make a prediction, change one variable, and write down why the result did or did not match your prediction.

## Exit ticket

**You are ready to continue when:** you can explain how depth remains trainable and choose a generalization intervention based on train/validation evidence.

**Next:** 28 — Text, Unicode, characters, words, and bytes.
